# Week 2: Feature Engineering & Data Preprocessing for Machine Learning

**AnalystLab Africa · Data Science Internship** · **Author:** Molemi Molokoane

**Dataset:** [Heart Failure Prediction](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction) (fedesoriano, Kaggle, ODbL licence). 918 patients pooled from five hospital datasets (Cleveland, Hungary, Switzerland, Long Beach VA, Stalog).

**Goal:** turn the raw patient records into a clean, encoded, scaled, machine-learning-ready dataset for predicting whether a patient has heart disease, and document every preprocessing decision.

### Workflow
1. Business understanding
2. Data inspection
3. Exploratory visualisation
4. Data cleaning
5. Feature engineering
6. Train/test split
7. Outlier detection and treatment
8. Feature encoding
9. Feature scaling
10. Feature selection
11. Final machine-learning dataset
12. Answers to the business questions

## 0. Setup

The notebook runs both in Google Colab and locally. If `Data/heart.csv` is not next to the notebook (for example in a fresh Colab session), the raw file is read from the project's GitHub repository.

In [ ]:
# Core libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn preprocessing and feature-selection tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

# Display and plotting defaults
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42  # fixed seed so every run gives the same split and importances

# Paths: outputs go in folders next to the notebook
DATA_DIR, VIZ_DIR = "Data", "Visualizations"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(VIZ_DIR, exist_ok=True)

LOCAL_CSV = os.path.join(DATA_DIR, "heart.csv")
GITHUB_CSV = ("https://raw.githubusercontent.com/Mo-lemi/AnalystLab-Africa-Work/main/"
              "Week2_DataScience_Project/Data/heart.csv")


def save_fig(name):
    """Save the current figure to the Visualizations folder and show it."""
    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, f"{name}.png"), bbox_inches="tight")
    plt.show()

## 1. Business Understanding

Cardiovascular disease is the world's leading cause of death, responsible for roughly 17.9 million deaths a year (WHO). Many of these deaths are preventable if high-risk patients are identified early and referred for further tests or treatment.

- **Business problem:** clinics need a fast, low-cost way to flag patients who are likely to have heart disease, using measurements already taken at a routine visit.
- **Objective:** prepare a dataset from which a classification model can predict heart disease reliably.
- **Target variable:** `HeartDisease` (1 = heart disease, 0 = normal).
- **Why predictive modelling:** a model can screen every patient consistently and instantly, helping clinicians decide who to prioritise for further tests.
- **Expected impact:** earlier diagnosis, better use of scarce specialist time and equipment, and fewer emergency admissions.

The full discussion is in `Reports/Business_Understanding_Report`.

## 2. Data Inspection

### 2.1 Load the dataset and display the first 10 rows

In [ ]:
# Read the raw CSV, locally if available, otherwise from GitHub
source = LOCAL_CSV if os.path.exists(LOCAL_CSV) else GITHUB_CSV
df_raw = pd.read_csv(source)
print(f"Loaded from: {source}")
df_raw.head(10)

### 2.2 Structure: rows, columns and data types

In [ ]:
print(f"Rows: {df_raw.shape[0]}   Columns: {df_raw.shape[1]}\n")
df_raw.info()

In [ ]:
# Group the columns by kind to plan the preprocessing
numeric_cols = df_raw.select_dtypes(include="number").columns.drop("HeartDisease").tolist()
categorical_cols = df_raw.select_dtypes(include="object").columns.tolist()
print("Numeric columns    :", numeric_cols)
print("Categorical columns:", categorical_cols)
print("Target             : HeartDisease")
print("\nUnique values per column:")
df_raw.nunique()

### 2.3 Missing values

`isnull()` finds no gaps, but a value of **0** is impossible for `Cholesterol` or `RestingBP` in a living patient. These zeros are missing values in disguise, so they are counted separately.

In [ ]:
# Standard NaN check
missing = pd.DataFrame({
    "NaN count": df_raw.isnull().sum(),
    "Zero count": (df_raw == 0).sum(),
})
missing["Zero %"] = (missing["Zero count"] / len(df_raw) * 100).round(1)
missing

In [ ]:
# Zeros only mean "missing" for physiological measurements that cannot be 0
hidden_missing = {col: int((df_raw[col] == 0).sum()) for col in ["RestingBP", "Cholesterol"]}
print("Hidden missing values (recorded as 0):", hidden_missing)

# Does missing cholesterol relate to the target?
chol_zero = df_raw["Cholesterol"] == 0
print(f"\nHeart-disease rate when Cholesterol = 0 : {df_raw.loc[chol_zero, 'HeartDisease'].mean():.1%}")
print(f"Heart-disease rate when Cholesterol > 0 : {df_raw.loc[~chol_zero, 'HeartDisease'].mean():.1%}")

**Observation:** 172 patients (18.7%) have `Cholesterol = 0` and one patient has `RestingBP = 0`. Patients with a missing cholesterol reading have a much higher heart-disease rate, so the *fact that it is missing* carries information. These rows come mainly from the Switzerland hospital source, which did not record cholesterol. This is why section 4 keeps a missing-value flag rather than only filling in the gaps.

The 0 values in `FastingBS` (binary flag) and `Oldpeak` (ST depression of 0 mm is normal) are valid.

### 2.4 Duplicate records

In [ ]:
print("Exact duplicate rows:", df_raw.duplicated().sum())
print("Duplicates ignoring the target:", df_raw.drop(columns="HeartDisease").duplicated().sum())

### 2.5 Summary statistics

In [ ]:
df_raw.describe().T.round(2)

In [ ]:
df_raw.describe(include="object").T

In [ ]:
# Class balance of the target
target_counts = df_raw["HeartDisease"].value_counts().rename({0: "Normal (0)", 1: "Heart disease (1)"})
print(target_counts)
print(f"\nPositive class share: {df_raw['HeartDisease'].mean():.1%}")

**Inspection summary**

| Check | Result |
|---|---|
| Shape | 918 rows × 12 columns |
| Data types | 5 numeric (int/float), 5 text categorical, 1 binary flag (`FastingBS`) stored as int, binary target |
| Missing values | No NaNs, but 172 hidden zeros in `Cholesterol` and 1 in `RestingBP` |
| Duplicates | None |
| Target balance | 55% heart disease / 45% normal: mildly imbalanced, so stratified splitting is enough |
| Value ranges | `Oldpeak` has 13 negative values (ST elevation), which are clinically valid; `Cholesterol` reaches 603 mg/dl |